<a href="https://colab.research.google.com/github/FirstTheKing05/Webtech67/blob/main/lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
x = 8.0
lr = 0.1                      # ขนาดก้าว

for i in range(21):
    if i in (0, 1, 2, 3, 5, 10, 20):
        print(f"ก้าว {i:2d}  x = {x:7.4f}  ความสูง = {x*x:8.4f}")
    gradient = 2*x            # ความชันใต้ฝ่าเท้า
    x = x - lr*gradient       # ก้าวสวนทางความชัน

ก้าว  0  x =  8.0000  ความสูง =  64.0000
ก้าว  1  x =  6.4000  ความสูง =  40.9600
ก้าว  2  x =  5.1200  ความสูง =  26.2144
ก้าว  3  x =  4.0960  ความสูง =  16.7772
ก้าว  5  x =  2.6214  ความสูง =   6.8719
ก้าว 10  x =  0.8590  ความสูง =   0.7379
ก้าว 20  x =  0.0922  ความสูง =   0.0085


In [2]:
for lr in [0.01, 0.1, 0.45, 0.55, 1.05]:
    x = 8.0
    for _ in range(20):
        x = x - lr*2*x
    print(f"lr {lr:5.2f}  หลัง 20 ก้าว x = {x:.4f}")

lr  0.01  หลัง 20 ก้าว x = 5.3409
lr  0.10  หลัง 20 ก้าว x = 0.0922
lr  0.45  หลัง 20 ก้าว x = 0.0000
lr  0.55  หลัง 20 ก้าว x = 0.0000
lr  1.05  หลัง 20 ก้าว x = 53.8200


In [6]:
import numpy as np

rng = np.random.default_rng(1)
X = np.array([[0,0],[0,1],[1,0],[1,1]], float)
y = np.array([[0],[1],[1],[0]], float)

W1 = rng.normal(0, 1, (2,2)); b1 = np.zeros((1,2))
W2 = rng.normal(0, 1, (2,1)); b2 = np.zeros((1,1))
sig = lambda z: 1/(1+np.exp(-z))

h = sig(X@W1 + b1)            # ชั้นซ่อนคิด
out = sig(h@W2 + b2)          # ชั้นตอบคิด
print("คำตอบก่อนฝึก", np.round(out.ravel(), 3))
print("loss ก่อนฝึก", round(float(np.mean((out-y)**2)), 4))

คำตอบก่อนฝึก [0.663 0.651 0.699 0.684]
loss ก่อนฝึก 0.2799


In [7]:
lr = 0.5
for epoch in range(20001):
    h = sig(X@W1 + b1)                  # จังหวะ 1 คิดไปข้างหน้า
    out = sig(h@W2 + b2)
    loss = np.mean((out-y)**2)          # จังหวะ 2 วัดความผิด
    if epoch in (0, 100, 1000, 5000, 20000):
        print(f"รอบ {epoch:5d}  loss = {loss:.4f}")

    d_out = (out-y)*out*(1-out)         # จังหวะ 3 ใบตำหนิชั้นตอบ
    d_h = d_out@W2.T * h*(1-h)          # ส่งย้อนไปชั้นซ่อน
    W2 -= lr*h.T@d_out; b2 -= lr*d_out.sum(0)
    W1 -= lr*X.T@d_h;  b1 -= lr*d_h.sum(0)

print("คำตอบหลังฝึก", np.round(out.ravel(), 3))

รอบ     0  loss = 0.2799
รอบ   100  loss = 0.2486
รอบ  1000  loss = 0.0155
รอบ  5000  loss = 0.0007
รอบ 20000  loss = 0.0001
คำตอบหลังฝึก [0.013 0.989 0.989 0.011]


In [8]:
print(np.round(sig(X@W1 + b1), 2))

[[0.04 0.03]
 [0.93 0.  ]
 [0.   0.95]
 [0.03 0.02]]


In [9]:
import torch
import torch.nn as nn

torch.manual_seed(1)
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y = torch.tensor([[0.],[1.],[1.],[0.]])

model = nn.Sequential(nn.Linear(2,2), nn.Sigmoid(),
                      nn.Linear(2,1), nn.Sigmoid())
opt = torch.optim.SGD(model.parameters(), lr=0.5)
loss_fn = nn.MSELoss()

for epoch in range(20001):
    out = model(X)                      # คิดไปข้างหน้า
    loss = loss_fn(out, y)              # วัดความผิด
    if epoch in (0, 100, 1000, 5000, 20000):
        print(f"รอบ {epoch:5d}  loss = {loss.item():.4f}")
    opt.zero_grad()
    loss.backward()                     # ใบตำหนิย้อนกลับ ในบรรทัดเดียว
    opt.step()                          # ขยับทุกน้ำหนักหนึ่งก้าว

print("คำตอบหลังฝึก", model(X).detach().numpy().round(3).ravel())

รอบ     0  loss = 0.2548
รอบ   100  loss = 0.2501
รอบ  1000  loss = 0.2500
รอบ  5000  loss = 0.0355
รอบ 20000  loss = 0.0004
คำตอบหลังฝึก [0.021 0.982 0.982 0.019]


In [10]:
def train_xor(lr, epochs=20000, seed=1):
    rng = np.random.default_rng(seed)
    X = np.array([[0,0],[0,1],[1,0],[1,1]], float)
    y = np.array([[0],[1],[1],[0]], float)
    W1 = rng.normal(0,1,(2,2)); b1 = np.zeros((1,2))
    W2 = rng.normal(0,1,(2,1)); b2 = np.zeros((1,1))
    sig = lambda z: 1/(1+np.exp(-z))
    for _ in range(epochs):
        h = sig(X@W1+b1); out = sig(h@W2+b2)
        d_out = (out-y)*out*(1-out)
        d_h = d_out@W2.T * h*(1-h)
        W2 -= lr*h.T@d_out; b2 -= lr*d_out.sum(0)
        W1 -= lr*X.T@d_h;  b1 -= lr*d_h.sum(0)
    h = sig(X@W1+b1); out = sig(h@W2+b2)
    return float(np.mean((out-y)**2)), np.round(out.ravel(), 3)

for lr in [0.01, 0.5, 5.0, 20.0]:
    L, preds = train_xor(lr)
    print(f"lr {lr:5.2f}  loss = {L:.4f}  คำตอบ = {preds}")

lr  0.01  loss = 0.2157  คำตอบ = [0.441 0.404 0.673 0.454]
lr  0.50  loss = 0.0001  คำตอบ = [0.013 0.989 0.989 0.011]
lr  5.00  loss = 0.0000  คำตอบ = [0.004 0.997 0.997 0.003]
lr 20.00  loss = 0.1443  คำตอบ = [0.    0.303 0.998 0.303]


Let's trace the value of `x` for `lr = 0.55`:

In [11]:
x = 8.0
lr = 0.55

print(f"Learning Rate (lr) = {lr}")
for i in range(21):
    print(f"Step {i:2d}: x = {x:10.7f}")
    gradient = 2*x
    x = x - lr*gradient

Learning Rate (lr) = 0.55
Step  0: x =  8.0000000
Step  1: x = -0.8000000
Step  2: x =  0.0800000
Step  3: x = -0.0080000
Step  4: x =  0.0008000
Step  5: x = -0.0000800
Step  6: x =  0.0000080
Step  7: x = -0.0000008
Step  8: x =  0.0000001
Step  9: x = -0.0000000
Step 10: x =  0.0000000
Step 11: x = -0.0000000
Step 12: x =  0.0000000
Step 13: x = -0.0000000
Step 14: x =  0.0000000
Step 15: x = -0.0000000
Step 16: x =  0.0000000
Step 17: x = -0.0000000
Step 18: x =  0.0000000
Step 19: x = -0.0000000
Step 20: x =  0.0000000


Now, let's trace the value of `x` for `lr = 1.05`:

In [12]:
x = 8.0
lr = 1.05

print(f"Learning Rate (lr) = {lr}")
for i in range(21):
    print(f"Step {i:2d}: x = {x:10.7f}")
    gradient = 2*x
    x = x - lr*gradient

Learning Rate (lr) = 1.05
Step  0: x =  8.0000000
Step  1: x = -8.8000000
Step  2: x =  9.6800000
Step  3: x = -10.6480000
Step  4: x = 11.7128000
Step  5: x = -12.8840800
Step  6: x = 14.1724880
Step  7: x = -15.5897368
Step  8: x = 17.1487105
Step  9: x = -18.8635815
Step 10: x = 20.7499397
Step 11: x = -22.8249336
Step 12: x = 25.1074270
Step 13: x = -27.6181697
Step 14: x = 30.3799867
Step 15: x = -33.4179854
Step 16: x = 36.7597839
Step 17: x = -40.4357623
Step 18: x = 44.4793385
Step 19: x = -48.9272724
Step 20: x = 53.8199996


### Explanation:

For `lr = 0.55`:
Even though `x` oscillates, its magnitude decreases with each step. This is because the update rule `x = x - lr * 2 * x` can be rewritten as `x = x * (1 - 2 * lr)`. When `lr = 0.55`, `1 - 2 * lr = 1 - 2 * 0.55 = 1 - 1.1 = -0.1`. So, `x` is multiplied by `-0.1` at each step. This means `x` alternates sign, but its absolute value shrinks (`|x_new| = 0.1 * |x_old|`), eventually converging to 0.

For `lr = 1.05`:
When `lr = 1.05`, `1 - 2 * lr = 1 - 2 * 1.05 = 1 - 2.1 = -1.1`. In this case, `x` is multiplied by `-1.1` at each step. This causes `x` to alternate signs, but its absolute value *increases* (`|x_new| = 1.1 * |x_old|`), leading to divergence. The steps are too large, overshooting the minimum with an ever-increasing magnitude.

### คำตอบ

1.  **สามจังหวะของการฝึกเครือข่ายประสาทคืออะไร และในโค้ด PyTorch ของด่านที่ 6 แต่ละจังหวะอยู่บรรทัดไหน**
    สามจังหวะหลักของการฝึกเครือข่ายประสาทคือ:
    *   **คิดไปข้างหน้า (Forward Pass):** การส่งข้อมูลผ่านเครือข่ายเพื่อคำนวณผลลัพธ์ของโมเดล
        *   ในโค้ด PyTorch (`loBs3HV-Qc8G`): `out = model(X)`
    *   **วัดความผิด (Loss Calculation):** การเปรียบเทียบผลลัพธ์ของโมเดลกับค่าจริง เพื่อหาขนาดของความผิดพลาด (loss)
        *   ในโค้ด PyTorch (`loBs3HV-Qc8G`): `loss = loss_fn(out, y)`
    *   **ใบตำหนิย้อนกลับ (Backward Pass/Optimization):** การคำนวณ Gradient ของ Loss เทียบกับน้ำหนักของโมเดล (Backward Pass) และการปรับน้ำหนักเหล่านั้นเพื่อลด Loss (Optimization)
        *   ในโค้ด PyTorch (`loBs3HV-Qc8G`): `loss.backward()` (คำนวณ Gradient) และ `opt.step()` (ปรับน้ำหนัก)

2.  **โมเดลภาษาใหญ่ฝึกด้วยกลไกเดียวกับสมองจิ๋วของเรา จากสิ่งที่เห็นในแล็บนี้ อธิบายว่าทำไมโมเดลที่ฝึกแบบนี้ถึงตอบเก่งมาก แต่ก็ตอบผิดอย่างมั่นใจได้ด้วย (บอกใบ้ ตลอดการฝึก เราถามมันว่าอะไร และไม่เคยถามว่าอะไร)**
    โมเดลภาษาใหญ่ (LLMs) ฝึกฝนโดยการเรียนรู้รูปแบบทางสถิติจากชุดข้อมูลขนาดมหาศาล เพื่อทำนายคำถัดไปที่น่าจะเป็นไปได้มากที่สุดในบริบทหนึ่งๆ ทำให้พวกมันสามารถสร้างข้อความที่ไหลลื่นและสอดคล้องกันได้ดีมาก
    
    อย่างไรก็ตาม ในระหว่างการฝึก เรา **'ถาม'** โมเดลว่าคำถัดไปคืออะไร โดยการพยายามลดความผิดพลาด (Loss) ระหว่างคำที่ทำนายกับคำจริง เรา **'ไม่ได้ถาม'** โมเดลว่าสิ่งที่ทำนายนั้นเป็นความจริง ถูกต้องตามข้อเท็จจริง หรือมีความเข้าใจเชิงลึกหรือไม่
    
    ดังนั้น ถ้าหากข้อมูลที่ใช้ฝึกมีอคติ ข้อผิดพลาด หรือมีรูปแบบทางสถิติที่นำไปสู่การทำนายที่ผิดพลาดแต่มีโอกาสเกิดสูง โมเดลก็จะตอบผิดอย่างมั่นใจได้ เพราะมันกำลังทำตามรูปแบบที่เรียนรู้มาโดยไม่มีกลไกในการ 'รู้' ว่าข้อมูลนั้นไม่ถูกต้องหรือเป็นเรื่องสมมติเหมือนมนุษย์

3.  **ถ้าฝึกโมเดลแล้ว loss พุ่งขึ้นเรื่อยๆ แทนที่จะลด สาเหตุแรกที่ควรสงสัยคืออะไร และจะลองแก้ยังไง**
    ถ้าฝึกโมเดลแล้วค่า Loss พุ่งขึ้นเรื่อยๆ แทนที่จะลดลง สาเหตุแรกที่ควรสงสัยคือ **อัตราการเรียนรู้ (Learning Rate) สูงเกินไป**
    
    *   **สาเหตุ:** อัตราการเรียนรู้ที่สูงเกินไปทำให้ Optimizer ปรับค่าน้ำหนักของโมเดลมากเกินไปในแต่ละขั้นตอน ทำให้ข้ามจุดต่ำสุดของฟังก์ชัน Loss ไปมาและทำให้ค่า Loss แทนที่จะลดลงกลับเพิ่มขึ้น หรือเกิดการแกว่งตัวที่ไม่เสถียร
    *   **วิธีแก้:** ลองลดอัตราการเรียนรู้ให้มีค่าน้อยลง (เช่น จาก `0.1` เป็น `0.01` หรือ `0.001`) เพื่อให้การปรับค่าน้ำหนักเป็นไปอย่างละเอียดขึ้นและค่อยๆ เข้าใกล้จุดต่ำสุดของฟังก์ชัน Loss